# 035 — Evaluation: the `combined_loss` alpha sweep

Compares the `models/alpha_sweep/alpha_<a>/<arch>/` checkpoints trained by
`025_training_alpha_sweep.ipynb` against each other and against `020`'s
default-alpha checkpoint (`models/deterministic/<arch>/`, `alpha =
settings.LOSS_ALPHA`), on the same axes as `033`/`040`:

- **reconstruction fidelity** — `mae`/`ssim`/`psnr` on the held-out test set;
- **detection** — AUROC against the hand-drawn masks (`GT01`-`GT03`);
- **stroke coherence** — the reference-free corroboration.

The comparison is **per architecture**: for each architecture, all of its
trained alphas side by side, so the question is *"where should this
architecture's fidelity/structure trade-off sit?"*, not *"which architecture
is best"* (that is `030`/`033`/`040`).

Two controls make the numbers readable:

- **`gray`** — `mean(R, G, B)` scored as if it were a model. The floor any
  trained checkpoint has to clear; also a direct read on how good an IR
  prior the visible grey level is on these paintings.
- **residual-head diagnostics** (`unet_residual` only) — `resid_l1 =
  mean |mu - gray|` (near zero => the `tanh` head collapsed to the
  identity) and `clip_frac` (fraction of pixels pinned at 0/1 by the
  straight-through clip).

**Read with `024` §6's confound in mind**: across `C0`+`C1`, detection
AUROC correlates with *worse* reconstruction (Spearman `rho = +0.665`), so a
column that wins fidelity and loses detection is the expected pattern, not a
contradiction.


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check. Importing `scripts.trainer.load_model` also registers `unet_residual`'s `RGBToGray`/`ClipToUnitStraightThrough` layers, which its checkpoints need to deserialize.

In [ ]:
import gc
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
    pad_to_multiple,
)
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Discover the runs

Every `models/alpha_sweep/alpha_<a>/<arch>/best_model.keras`, plus `020`'s
`models/deterministic/<arch>/` as the `alpha = settings.LOSS_ALPHA`
baseline for each architecture that has a sweep run. Set `SELECTED_ARCHS`
to restrict the comparison to specific architectures; leave it `None` for
all of them.

In [ ]:
SELECTED_ARCHS: list[str] | None = None  # e.g. ["unet_residual"] to focus

DET_DIR = settings.MODELS_DIR / "deterministic"
SWEEP_DIR = settings.MODELS_DIR / "alpha_sweep"


class Run(NamedTuple):
    arch: str
    alpha: float
    model_dir: Path
    is_baseline: bool

    @property
    def label(self) -> str:
        tag = "020" if self.is_baseline else f"a={self.alpha:.2f}"
        return f"{self.arch} [{tag}]"

    @property
    def checkpoint(self) -> Path:
        return self.model_dir / self.arch / "best_model.keras"


sweep_archs: dict[str, list[Run]] = {}
for adir in sorted(SWEEP_DIR.glob("alpha_*")):
    try:
        alpha = float(adir.name.removeprefix("alpha_"))
    except ValueError:
        continue
    for ckpt in sorted(adir.glob("*/best_model.keras")):
        arch = ckpt.parent.name
        sweep_archs.setdefault(arch, []).append(Run(arch, alpha, adir, False))

if SELECTED_ARCHS is not None:
    sweep_archs = {a: r for a, r in sweep_archs.items() if a in SELECTED_ARCHS}

RUNS: list[Run] = []
for arch, runs in sorted(sweep_archs.items()):
    base = Run(arch, settings.LOSS_ALPHA, DET_DIR, True)
    if base.checkpoint.exists():
        RUNS.append(base)
    RUNS.extend(sorted(runs, key=lambda r: r.alpha))

if not RUNS:
    raise RuntimeError(
        "No alpha-sweep checkpoints under models/alpha_sweep/ — run 025 first."
    )

ARCHS = sorted({r.arch for r in RUNS})
for arch in ARCHS:
    labels = [r.label for r in RUNS if r.arch == arch]
    print(f"{arch:<18} {labels}")

## 2. Test dataset

Artwork-and-mockups split, same as every training notebook — the `mae`/
`ssim`/`psnr` below are on data none of these checkpoints saw.

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
_, _, test_pairs = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)
test_ds = build_dataset(test_pairs, batch_size=settings.BATCH_SIZE, augment=False, shuffle=False)
print(f"Test pairs: {len(test_pairs)} | batches: {len(test_ds)}")

## 3. Ground-truth images

Discovered from `data/test/annotations/*_Map.png` — `GT01`-`GT03`.
Detection AUROC and stroke coherence are scored on these; the `gray`
control is scored here too.

In [ ]:
TEST_RGB = project_root / "data" / "test" / "rgb"
TEST_IR = project_root / "data" / "test" / "ir"
ANN = project_root / "data" / "test" / "annotations"
MASK_THRESHOLD = 127
CLIP_EPS = 1e-6

gt_pairs = []
for mp in sorted(ANN.glob("*_Map.png")):
    stem = mp.name.removesuffix("_Map.png")
    rgb_p, ir_p = TEST_RGB / f"{stem}.jpg", TEST_IR / f"{stem}.jpg"
    if rgb_p.exists() and ir_p.exists():
        gt_pairs.append((stem, rgb_p, ir_p))
GT_STEMS = [s for s, _, _ in gt_pairs]
print(f"Ground-truth images: {GT_STEMS}")
if len(gt_pairs) < 2:
    raise RuntimeError("Need >= 2 ground-truth masks for the per-image breakdown.")


def load_pair(rgb_p, ir_p):
    rgb = np.array(Image.open(rgb_p).convert("RGB"), np.float32) / 255.0
    ir = np.array(Image.open(ir_p).convert("L"), np.float32) / 255.0
    return rgb, ir


def load_mask(stem):
    return np.array(Image.open(ANN / f"{stem}_Map.png").convert("L")) > MASK_THRESHOLD


def predict_mu(model, rgb, hw):
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = hw
    return model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, 0]


def signals_of(ir, mu):
    d = analyze_delta(ir, mu)
    return {"raw delta": d.raw_delta, "structural delta": d.structural_delta}


def fidelity(ir, mu):
    real, pred = tf.constant(ir[..., None]), tf.constant(mu[..., None])
    return {
        "mae": float(np.mean(np.abs(ir - mu))),
        "ssim": float(tf.image.ssim(real, pred, max_val=1.0)),
        "psnr": float(tf.image.psnr(real, pred, max_val=1.0)),
    }

### The `gray` control

`mean(R, G, B)` scored as a model — fidelity on the test set, detection/
coherence on the ground-truth images. Every trained column is read against
this row first.

In [ ]:
# fidelity of the gray baseline on the full test set
gray_test_fid = {m: [] for m in ("mae", "ssim", "psnr")}
for rgb_b, ir_b in test_ds.unbatch().as_numpy_iterator():
    g = rgb_b.mean(axis=-1)
    for m, v in fidelity(ir_b[..., 0], g).items():
        gray_test_fid[m].append(v)
gray_test_fid = {m: float(np.mean(v)) for m, v in gray_test_fid.items()}
print("gray (test set):", {m: f"{v:.4f}" for m, v in gray_test_fid.items()})

# detection / coherence of the gray baseline on the ground-truth images
gray_auroc = {k: {} for k in ("raw delta", "structural delta")}
gray_coh = {k: [] for k in ("raw delta", "structural delta")}
for stem, rgb_p, ir_p in gt_pairs:
    rgb, ir = load_pair(rgb_p, ir_p)
    g = rgb.mean(axis=-1)
    mask = load_mask(stem)
    for kind, sig in signals_of(ir, g).items():
        gray_auroc[kind][stem] = evaluate_detection(sig, mask).auroc
        gray_coh[kind].append(stroke_coherence(sig).coherence)
print("gray structural-delta AUROC:",
      {k: f"{v:.3f}" for k, v in gray_auroc["structural delta"].items()})

## 4. The pass

One run at a time: load, `evaluate` on the test set for pixel fidelity,
predict each ground-truth image for the signals, then release before the
next (checkpoints are ~370 MB). `resid_l1`/`clip_frac` are recorded only
for `unet_residual`.

In [ ]:
fid_rows: dict[str, dict[str, float]] = {}
diag_rows: dict[str, dict[str, float]] = {}
auroc_rows: dict[tuple[str, str], dict[str, float]] = {}
coh_rows: dict[tuple[str, str], list[float]] = {}

for run in RUNS:
    try:
        model = load_model(run.arch, model_dir=run.model_dir, loss_alpha=run.alpha)
    except FileNotFoundError as exc:
        print(f"[skip] {run.label}: {exc}")
        continue
    print(f"=== {run.label} ===")

    m = model.evaluate(test_ds, verbose=0, return_dict=True)
    fid_rows[run.label] = {k: float(v) for k, v in m.items()}

    resid_l1, clip_frac = [], []
    for stem, rgb_p, ir_p in gt_pairs:
        rgb, ir = load_pair(rgb_p, ir_p)
        mu = predict_mu(model, rgb, ir.shape)
        for kind, sig in signals_of(ir, mu).items():
            auroc_rows.setdefault((run.label, kind), {})[stem] = evaluate_detection(sig, load_mask(stem)).auroc
            coh_rows.setdefault((run.label, kind), []).append(stroke_coherence(sig).coherence)
        if run.arch == "unet_residual":
            g = rgb.mean(axis=-1)
            resid_l1.append(float(np.mean(np.abs(mu - g))))
            clip_frac.append(float(np.mean((mu <= CLIP_EPS) | (mu >= 1.0 - CLIP_EPS))))
    if resid_l1:
        diag_rows[run.label] = {"resid_l1": float(np.mean(resid_l1)),
                                "clip_frac": float(np.mean(clip_frac))}

    del model
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nEvaluated: {list(fid_rows)}")

## 5. Reconstruction fidelity

`mae`/`ssim`/`psnr` on the test set, per run, grouped by architecture. The
`combined_loss` value itself is *not* comparable across alphas (different
loss function) so it is omitted; `mae`/`ssim`/`psnr` are alpha-independent.
The `gray` row is the parameter-free floor.

In [ ]:
PIX = ["mae", "ssim", "psnr"]  # exact model.evaluate metric names (scripts/trainer.py)

w = 16
print("run".ljust(22) + "".join(k.rjust(w) for k in PIX))
print("gray".ljust(22) + "".join(f"{gray_test_fid[k]:.4f}".rjust(w) for k in PIX))
print("-" * (22 + w * len(PIX)))
for arch in ARCHS:
    for run in [r for r in RUNS if r.arch == arch]:
        vals = fid_rows.get(run.label)
        if vals is None:
            continue
        print(run.label.ljust(22) + "".join(f"{vals[k]:.4f}".rjust(w) for k in PIX))
    print()

## 6. Residual-head diagnostics (`unet_residual` only)

`resid_l1 = mean |mu - gray|` near zero, together with `mae` close to the
`gray` control's, means the `tanh` head collapsed to the identity — 31M
parameters reproducing a channel mean. `fixing.md` recorded exactly this
at `alpha = 0.16`.

In [ ]:
if diag_rows:
    print("run".ljust(22) + "resid_l1".rjust(11) + "clip_frac".rjust(11)
          + "mae".rjust(10) + "   note")
    print("-" * 66)
    for label, dd in diag_rows.items():
        mae = fid_rows.get(label, {}).get("mae", float("nan"))
        note = "COLLAPSED to gray" if dd["resid_l1"] < 0.01 else "residual in use"
        print(label.ljust(22) + f"{dd['resid_l1']:.4f}".rjust(11)
              + f"{dd['clip_frac']:.4f}".rjust(11) + f"{mae:.4f}".rjust(10)
              + f"   {note}")
    print(f"\n(gray control mae = {gray_test_fid['mae']:.4f})")
else:
    print("No unet_residual run in the selection - nothing to diagnose.")

## 7. Detection — AUROC against the masks

Mean over `GT01`-`GT03`, per signal, with the `gray` control. `0.5` is
chance. A column at or below `gray` has added nothing to detection whatever
its fidelity.

In [ ]:
for kind in ("structural delta", "raw delta"):
    print(f"\n=== {kind} ===")
    print("run".ljust(22) + "".join(s.rjust(9) for s in GT_STEMS) + "mean".rjust(9))
    gv = gray_auroc[kind]
    print("gray".ljust(22) + "".join(f"{gv[s]:.3f}".rjust(9) for s in GT_STEMS)
          + f"{np.mean(list(gv.values())):.3f}".rjust(9))
    print("-" * (22 + 9 * (len(GT_STEMS) + 1)))
    for arch in ARCHS:
        for run in [r for r in RUNS if r.arch == arch]:
            key = (run.label, kind)
            if key not in auroc_rows:
                continue
            d = auroc_rows[key]
            print(run.label.ljust(22) + "".join(f"{d[s]:.3f}".rjust(9) for s in GT_STEMS)
                  + f"{np.mean(list(d.values())):.3f}".rjust(9))
        print()

## 8. Stroke coherence — reference-free corroboration

How much of each signal is oriented, line-like structure rather than
isotropic noise (`scripts.stroke_stats`). Structurally independent from the
AUROC above; a valid tie-breaker between similar-AUROC columns, not a
primary axis.

In [ ]:
for kind in ("structural delta", "raw delta"):
    print(f"\n=== {kind} ===")
    print("run".ljust(22) + "coherence (mean +/- std over GT)".rjust(34))
    gc_ = np.array(gray_coh[kind])
    print("gray".ljust(22) + f"{gc_.mean():.3f} +/- {gc_.std():.3f}".rjust(34))
    print("-" * 56)
    for arch in ARCHS:
        for run in [r for r in RUNS if r.arch == arch]:
            key = (run.label, kind)
            if key not in coh_rows:
                continue
            v = np.array(coh_rows[key])
            print(run.label.ljust(22) + f"{v.mean():.3f} +/- {v.std():.3f}".rjust(34))
        print()

## 9. Verdict — per architecture

For each architecture: the alpha that wins pixel fidelity (`mae`), the
alpha that wins detection (`structural delta` AUROC), and whether either
beats the `020` default and the `gray` floor. Different winners on the two
axes is the `024` §6 confound, not a contradiction — decide which axis the
deliverable cares about (detection) before reading this as a
recommendation.

In [ ]:
def _mae(label):
    return fid_rows.get(label, {}).get("mae", float("inf"))


def _auroc(label):
    d = auroc_rows.get((label, "structural delta"), {})
    return float(np.mean(list(d.values()))) if d else float("-inf")


gray_sd = float(np.mean(list(gray_auroc["structural delta"].values())))
print(f"gray: mae={gray_test_fid['mae']:.4f}  structural-delta AUROC={gray_sd:.3f}\n")

for arch in ARCHS:
    runs = [r for r in RUNS if r.arch == arch and r.label in fid_rows]
    if not runs:
        continue
    best_fid = min(runs, key=lambda r: _mae(r.label))
    best_det = max(runs, key=lambda r: _auroc(r.label))
    base = next((r for r in runs if r.is_baseline), None)
    print(f"{arch}")
    print(f"  best fidelity : {best_fid.label:<18} mae={_mae(best_fid.label):.4f}")
    print(f"  best detection: {best_det.label:<18} structural-delta AUROC={_auroc(best_det.label):.3f}"
          + ("" if _auroc(best_det.label) > gray_sd + 0.01 else "   (<= gray, no real detection)"))
    if base is not None:
        print(f"  vs 020 default: mae {_mae(base.label):.4f} -> {_mae(best_fid.label):.4f}, "
              f"AUROC {_auroc(base.label):.3f} -> {_auroc(best_det.label):.3f}")
    print()